# 06 - Comparison & Analysis

Brings every model's result into one table and runs the deeper analyses that turn numbers into findings:

1. the headline metric table (all models, same metrics),
2. metrics **sliced by trip length** (does attention help on longer trips?),
3. metrics **sliced by destination popularity** (head vs tail),
4. notes on **multi-seed** spread and **efficiency**.

Rows come from `results.csv`, which every model notebook appends to. The sliced analyses re-run our two models (FPMC, SASRec) to obtain per-trip ranked lists; teammates can add Markov / GRU4Rec the same way.

In [ ]:
import os, sys
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))
ROOT = find_root(); sys.path.insert(0, ROOT)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.preprocessing import load_processed
data = load_processed(os.path.join(ROOT, 'data', 'processed'))

## 1. Headline table (from results.csv)

In [ ]:
res = pd.read_csv(os.path.join(ROOT, 'results.csv')).drop_duplicates('model', keep='last')
res = res.set_index('model').sort_values('NDCG@10')
display(res.round(4))
res[['HR@10', 'NDCG@10', 'MRR']].plot.barh(figsize=(8, 4)); plt.title('Model comparison'); plt.tight_layout(); plt.show()

## 2 & 3. Sliced analyses (re-run our two models for per-trip rankings)

In [ ]:
import torch
from src.models.fpmc import run_fpmc
from src.models.sasrec import run_sasrec
from src.evaluate import evaluate_sliced, city_frequency
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

_, fpmc_ranked, _ = run_fpmc(data, axis='user', epochs=20, verbose=False)
_, sas_ranked, _ = run_sasrec(data, side_feats=(), epochs=30, device=DEVICE, verbose=False)
targets = [t['target_city'] for t in data['test']]
ranked_by_model = {'FPMC-user': fpmc_ranked, 'SASRec': sas_ranked}

In [ ]:
# --- by trip length (observed prefix length) ---
def length_group(t):
    n = len(t['cities'])
    return '<=3' if n <= 3 else ('4-5' if n <= 5 else '6+')
groups = [length_group(t) for t in data['test']]
for name, ranked in ranked_by_model.items():
    sl = evaluate_sliced(ranked, targets, groups)
    print(name, {g: round(m['NDCG@10'], 4) for g, m in sl.items()})

In [ ]:
# --- by destination popularity (head / mid / tail by train frequency) ---
freq = city_frequency(data)
q50, q90 = np.quantile(freq[freq > 0], [0.5, 0.9])
def pop_group(tgt):
    f = freq[tgt]
    return 'head' if f >= q90 else ('tail' if f <= q50 else 'mid')
pgroups = [pop_group(t) for t in targets]
for name, ranked in ranked_by_model.items():
    sl = evaluate_sliced(ranked, targets, pgroups)
    print(name, {g: round(m['NDCG@10'], 4) for g, m in sl.items()})

## 4. Rigor notes for the report

- **Multi-seed:** re-run each trained model (FPMC, SASRec) with 2-3 seeds and report mean +/- spread on the headline metrics, so a gap reflects the model, not a lucky initialisation.
- **Efficiency:** `results.csv` carries `train_secs` and `n_params` per model - report the accuracy-vs-cost trade-off.
- **Sanity peg:** Popularity's Accuracy@4 reproduces the repo (~0.053), confirming the scorer is correct.
- **Full-data note:** these are 5k-subsample, CPU numbers; the final report should rerun on the full dataset (SASRec on a Colab GPU).